In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


In [ ]:
import os

print("Nội dung /kaggle/input:")
for d in sorted(os.listdir("/kaggle/input")):
    print(" -", d)

DATASET_ROOT = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH"

print("\nDATASET_ROOT =", DATASET_ROOT)
if not os.path.isdir(DATASET_ROOT):
    print("Không tìm thấy DATASET_ROOT")
else:
    for sub in ["TRAIN/clean", "TRAIN/noise", "TEST/clean"]:
        p = os.path.join(DATASET_ROOT, sub)
        if os.path.isdir(p):
            n = sum(1 for _ in os.scandir(p))
            print(f"OK  {sub:15s} -> {n} mục")
        else:
            print(f"!!! {sub:15s} -> KHÔNG TỒN TẠI, kiểm tra lại cấu trúc dataset")


In [ ]:
import glob
import soundfile as sf

def check_native_sr(folder, n=5):
    files = glob.glob(os.path.join(folder, "**", "*.wav"), recursive=True)[:n]
    for f in files:
        info = sf.info(f)
        print(f"  {os.path.basename(f):30s} sr={info.samplerate}")

print("Sample rate của vài file CLEAN:")
check_native_sr(os.path.join(DATASET_ROOT, "TRAIN", "CLEAN"))
print("\nSample rate của vài file NOISE:")
check_native_sr(os.path.join(DATASET_ROOT, "TRAIN", "NOISE"))
print("\n=> Nếu sr in ra là 16000 (không phải 48000), dữ liệu train của bạn cũng bị")
print("   'băng thông giả' giống hệt inputTest lúc infer -> cần cân nhắc train model")
print("   ở sr=16000 thay vì ép 48000, hoặc thu âm lại dữ liệu gốc ở 48kHz thật.")


In [ ]:
!apt-get update -qq
!apt-get install -y -qq rustc cargo libhdf5-dev pkg-config patchelf > /dev/null
!pip install -q patchelf
!rustc --version && cargo --version


In [ ]:
%cd /kaggle/working
!rm -rf DeepFilterNet
!git clone --depth 1 --branch v0.5.6 https://github.com/Rikorose/DeepFilterNet.git
%cd /kaggle/working/DeepFilterNet
!git log -1 --oneline


In [ ]:
%pip install -q --force-reinstall torch==2.2.0 torchaudio==2.2.0
%pip install -q --force-reinstall "numpy<2"
%pip install -q maturin h5py soundfile librosa loguru appdirs requests packaging sympy icecream pystoi pesq scipy


In [ ]:
%cd /kaggle/working/DeepFilterNet
!mkdir -p /tmp/wheels
!maturin build --release -m pyDF/Cargo.toml -o /tmp/wheels
!maturin build --release -m pyDF-data/Cargo.toml -o /tmp/wheels
!pip install -q /tmp/wheels/*.whl


In [ ]:
import os, sys

REPO_DF = "/kaggle/working/DeepFilterNet/DeepFilterNet"
os.environ["PYTHONPATH"] = REPO_DF
sys.path.insert(0, REPO_DF)

import libdf
import libdfdata
print("libdf & libdfdata import OK")


In [ ]:
import os, zipfile, shutil

WORK = "/kaggle/working"
REPO = "/kaggle/working/DeepFilterNet"

base_dir = os.path.join(WORK, "finetune_base")
checkpoints_dir = os.path.join(base_dir, "checkpoints")
os.makedirs(checkpoints_dir, exist_ok=True)

PRETRAINED_ZIP = os.path.join(REPO, "models", "DeepFilterNet3.zip") 
extract_dst = os.path.join(WORK, "pretrained_extract")

if not os.path.isdir(extract_dst):
    with zipfile.ZipFile(PRETRAINED_ZIP) as zf:
        zf.extractall(extract_dst)

src = os.path.join(extract_dst, "DeepFilterNet3")
shutil.copy(os.path.join(src, "config.ini"), os.path.join(base_dir, "config.ini"))
for fn in os.listdir(os.path.join(src, "checkpoints")):
    shutil.copy(os.path.join(src, "checkpoints", fn), checkpoints_dir)

print("Đã chuẩn bị base model để finetune từ: DeepFilterNet3 (SOTA)")
print("base_dir:", base_dir)
print("checkpoints:", os.listdir(checkpoints_dir))


In [ ]:
import os
print("DATASET_ROOT =", DATASET_ROOT)
print("Tồn tại?", os.path.isdir(DATASET_ROOT))
print("--- Trong DATASET_ROOT ---")
print(os.listdir(DATASET_ROOT))
print("--- Trong TRAIN ---")
print(os.listdir(os.path.join(DATASET_ROOT, "TRAIN")))
print("--- Trong TRAIN/CLEAN (vài file đầu) ---")
print(os.listdir(os.path.join(DATASET_ROOT, "TRAIN", "CLEAN"))[:5])


In [ ]:

import glob, random

random.seed(42)
VALID_RATIO = 0.1
TEST_RATIO = 0.1

def list_wavs(d):
    return sorted(glob.glob(os.path.join(d, "**", "*.wav"), recursive=True))

speech_files = list_wavs(os.path.join(DATASET_ROOT, "TRAIN", "CLEAN"))
noise_files = list_wavs(os.path.join(DATASET_ROOT, "TRAIN", "NOISE"))
print(f"Tìm thấy {len(speech_files)} file speech (clean), {len(noise_files)} file noise trong TRAIN")
assert len(speech_files) > 0, "Không tìm thấy .wav trong TRAIN/clean — kiểm tra lại DATASET_ROOT"
assert len(noise_files) > 0, "Không tìm thấy .wav trong TRAIN/noise — kiểm tra lại DATASET_ROOT"

random.shuffle(speech_files)
random.shuffle(noise_files)

n_sp_val = max(1, int(len(speech_files) * VALID_RATIO))
n_sp_test = max(1, int(len(speech_files) * TEST_RATIO))
n_no_val = max(1, int(len(noise_files) * VALID_RATIO))
n_no_test = max(1, int(len(noise_files) * TEST_RATIO))

speech_valid = speech_files[:n_sp_val]
speech_test = speech_files[n_sp_val:n_sp_val + n_sp_test]
speech_train = speech_files[n_sp_val + n_sp_test:]

noise_valid = noise_files[:n_no_val]
noise_test = noise_files[n_no_val:n_no_val + n_no_test]
noise_train = noise_files[n_no_val + n_no_test:]

DATA_LIST_DIR = "/kaggle/working/data_lists"
os.makedirs(DATA_LIST_DIR, exist_ok=True)

def write_list(files, name):
    p = os.path.join(DATA_LIST_DIR, name)
    with open(p, "w") as f:
        f.write("\n".join(files))
    return p

write_list(speech_train, "speech_train.txt")
write_list(speech_valid, "speech_valid.txt")
write_list(speech_test, "speech_test.txt")
write_list(noise_train, "noise_train.txt")
write_list(noise_valid, "noise_valid.txt")
write_list(noise_test, "noise_test.txt")

print(f"Train: {len(speech_train)} speech / {len(noise_train)} noise")
print(f"Valid: {len(speech_valid)} speech / {len(noise_valid)} noise")
print(f"Test : {len(speech_test)} speech / {len(noise_test)} noise")


In [ ]:
import subprocess

DATA_HDF5_DIR = "/kaggle/working/data_hdf5"
os.makedirs(DATA_HDF5_DIR, exist_ok=True)

jobs = [
    ("speech", "speech_train.txt", "SPEECH_TRAIN.hdf5"),
    ("speech", "speech_valid.txt", "SPEECH_VALID.hdf5"),
    ("speech", "speech_test.txt", "SPEECH_TEST.hdf5"),
    ("noise", "noise_train.txt", "NOISE_TRAIN.hdf5"),
    ("noise", "noise_valid.txt", "NOISE_VALID.hdf5"),
    ("noise", "noise_test.txt", "NOISE_TEST.hdf5"),
]

env = os.environ.copy()
env["PYTHONPATH"] = REPO_DF

for kind, list_name, out_name in jobs:
    cmd = [
        "python", "df/scripts/prepare_data.py",
        "--sr", "48000",
        kind,
        os.path.join(DATA_LIST_DIR, list_name),
        os.path.join(DATA_HDF5_DIR, out_name),
    ]
    print("Đang chạy:", " ".join(cmd))
    subprocess.run(cmd, cwd=REPO_DF, env=env, check=True)

print("\nĐã tạo xong các file HDF5:")
for f in sorted(os.listdir(DATA_HDF5_DIR)):
    print(" -", f)


In [ ]:
import json

dataset_cfg = {
    "train": [["SPEECH_TRAIN.hdf5", 1.0], ["NOISE_TRAIN.hdf5", 1.0]],
    "valid": [["SPEECH_VALID.hdf5", 1.0], ["NOISE_VALID.hdf5", 1.0]],
    "test": [["SPEECH_TEST.hdf5", 1.0], ["NOISE_TEST.hdf5", 1.0]],
}

cfg_path = "/kaggle/working/dataset.cfg"
with open(cfg_path, "w") as f:
    json.dump(dataset_cfg, f, indent=2)

print("Đã tạo", cfg_path)
print(json.dumps(dataset_cfg, indent=2))


In [ ]:
import configparser

FT_EXTRA_EPOCHS = 20
FT_LR = 5e-5
FT_BATCH_SIZE = 8
FT_ATTEN_LIM_DB = 12  

ckpt_files = [
    f for f in os.listdir(checkpoints_dir)
    if f.startswith("model_") and (f.endswith(".ckpt") or f.endswith(".ckpt.best"))
]

def get_epoch(fn):
    return int(fn.split(".")[0].split("_")[-1])

cur_epoch = max(get_epoch(f) for f in ckpt_files) if ckpt_files else 0
print("Epoch hiện tại của checkpoint:", cur_epoch)

cp = configparser.ConfigParser()
cp.read(os.path.join(base_dir, "config.ini"))

if not cp.has_section("train"):
    cp.add_section("train")

cp["train"]["lr"] = str(FT_LR)                    
cp["train"]["max_epochs"] = str(cur_epoch + FT_EXTRA_EPOCHS)
cp["train"]["batch_size"] = str(FT_BATCH_SIZE)
cp["train"]["batch_size_eval"] = str(FT_BATCH_SIZE)
cp["train"]["p_atten_lim"] = str(FT_ATTEN_LIM_DB)  
if cp.has_option("train", "p_reverb"):
    cp["train"]["p_reverb"] = "0.0"         

with open(os.path.join(base_dir, "config.ini"), "w") as f:
    cp.write(f)

print(f"Sẽ finetune từ epoch {cur_epoch} tới epoch {cur_epoch + FT_EXTRA_EPOCHS}, "
      f"lr={FT_LR}, batch_size={FT_BATCH_SIZE}, p_atten_lim={FT_ATTEN_LIM_DB}dB")

print("\n--- Nội dung config.ini [train] sau khi sửa ---")
print(dict(cp["train"]))


In [ ]:
import subprocess

cmd = [
    "python", "df/train.py",
    "/kaggle/working/dataset.cfg",
    "/kaggle/working/data_hdf5",
    base_dir,
]

env = os.environ.copy()
env["PYTHONPATH"] = REPO_DF

with open("/kaggle/working/train.log", "w") as logf:
    proc = subprocess.Popen(cmd, cwd=REPO_DF, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        logf.write(line)
    proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"train.py thoát với mã lỗi {proc.returncode}, xem chi tiết ở /kaggle/working/train.log")
print("\nFinetune hoàn tất. Checkpoint mới nằm trong:", checkpoints_dir)


In [ ]:
import shutil

zip_path = shutil.make_archive("/kaggle/working/finetune_checkpoint", "zip", base_dir)
print("Đã tạo file zip:", zip_path)

import os
print("\nDung lượng:", os.path.getsize(zip_path) / 1e6, "MB")


In [ ]:
import glob, subprocess

test_files = sorted(glob.glob(os.path.join(DATASET_ROOT, "TEST ", "*.wav"), recursive=True))
print(f"Tìm thấy {len(test_files)} file trong TEST/clean")
assert len(test_files) > 0, "Không tìm thấy file .wav nào trong TEST/clean"

OUT_DIR = "/kaggle/working/enhanced_test"
os.makedirs(OUT_DIR, exist_ok=True)

cmd = [
    "python", "df/enhance.py",
    "-m", base_dir,
    "-o", OUT_DIR,
] + test_files

env = os.environ.copy()
env["PYTHONPATH"] = REPO_DF

subprocess.run(cmd, cwd=REPO_DF, env=env, check=True)
print("\nĐã enhance xong. Kết quả nằm ở:", OUT_DIR)
print("Số file output:", len(os.listdir(OUT_DIR)))


In [ ]:
import shutil

zip_path = shutil.make_archive("/kaggle/working/enhanced_test", "zip", "/kaggle/working/enhanced_test")
print("Đã tạo file zip:", zip_path)
print("Vào tab Output/Data ở panel bên phải của Kaggle để tải file này về.")